In [2]:
pip install numpy matplotlib Pillow reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.8 MB/s eta 0:00:00


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Round CFT (Encased RC) — Comprehensive P–M Envelope with Full Mathematical Derivations
----------------------------------------------------------------------------------------
• Concrete tension = 0; compression capped at 0.95 f'c (round CFT per AISC 360-22); εcu = 0.003
• Steel tube & rebars: elastic–perfectly plastic (Es–Fy)
• One-way shear (ACI 318-19) for solid circular core:
    Vc = 2·λ·√(f′c[psi]) · b_w · d, with d = 0.8·D and b_w = D (normalweight λ = 1.0)
• Outputs (per-run folder):
  - PM_points.csv (UTF-8 BOM)
  - Key_Points.csv (UTF-8 BOM)
  - pm_envelope.png (nominal envelope + cloud + 0.8·Pn,max + factored envelope per ACI 318 Table 21.2.2 + 0.8·ϕPn,max)
  - cft_section.png (section sketch)
  - Encased_Drilled_Pier_Report.pdf (full report with line-by-line mathematical proofs)

Features:
  - Professional title block with dynamic logo scaling
  - Line-by-line mathematical derivations for all capacity calculations
  - Full traceability of engineering assumptions and code references

Revised: 2025-01-XX
"""

from dataclasses import dataclass, field
from typing import Optional, Tuple, List, Dict, Any
import numpy as np
import math
import os
import csv
import time
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image as PILImage

# ---------- ReportLab (PDF) ----------
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                TableStyle, Image, PageBreak, KeepTogether,
                                HRFlowable, CondPageBreak)
from reportlab.lib.pagesizes import LETTER, A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.pdfgen import canvas

# ---------- Rebar Database (ASTM A615) ----------
BAR_AREAS: Dict[int, float] = {  # in²
    3: 0.11, 4: 0.20, 5: 0.31, 6: 0.44, 7: 0.60,
    8: 0.79, 9: 1.00, 10: 1.27, 11: 1.56, 14: 2.25, 18: 4.00
}
BAR_DIAM: Dict[int, float] = {  # in (nominal diameter)
    3: 0.375, 4: 0.500, 5: 0.625, 6: 0.750, 7: 0.875,
    8: 1.000, 9: 1.128, 10: 1.270, 11: 1.410, 14: 1.693, 18: 2.257
}

# ---------- Project Information Dataclass ----------
@dataclass
class ProjectInfo:
    """Project metadata for title block"""
    project_name: str = "Encased Drilled Pier Analysis"
    project_number: str = "----"
    client: str = "----"
    location: str = "----"
    engineer: str = "----"
    checker: str = "----"
    date: str = field(default_factory=lambda: time.strftime("%Y-%m-%d"))
    revision: str = "0"
    logo_path: Optional[str] = None  # Path to logo image (PNG, JPG, etc.)

# ---------- Materials Dataclass ----------
@dataclass
class Materials:
    fc: float        # ksi - concrete compressive strength
    Ec: float        # ksi - concrete modulus of elasticity
    Es: float        # ksi - steel modulus of elasticity
    Fy_tube: float   # ksi - steel tube yield strength
    fy_bar: float    # ksi - rebar yield strength
    eps_cu: float = 0.003           # ultimate concrete strain (ACI 318-19 §22.2.2.1)
    conf_cap: float = 0.95          # 0.95 f'c cap for round CFT (AISC 360-22 §I2.1b)
    lambda_nw: float = 1.0          # ACI density factor λ (1.0 = normalweight, ACI 318-19 §19.2.4)
    wc: float = 150.0               # concrete unit weight (pcf)

    def __post_init__(self):
        """Validate material properties"""
        if self.fc <= 0:
            raise ValueError("f'c must be positive")
        if self.Fy_tube <= 0:
            raise ValueError("Fy_tube must be positive")
        if self.fy_bar <= 0:
            raise ValueError("fy_bar must be positive")

# ---------- Geometry Dataclass ----------
@dataclass
class Geometry:
    D_core: float       # in - concrete core diameter (inside of tube)
    t_tube: float       # in - steel tube wall thickness
    cover: float        # in - clear cover to bar centerline
    n_bars: int         # number of longitudinal reinforcing bars
    bar_size: int       # bar designation (#3 through #18)

    def __post_init__(self):
        """Validate geometry and compute derived properties"""
        if self.D_core <= 0:
            raise ValueError("D_core must be positive")
        if self.t_tube <= 0:
            raise ValueError("t_tube must be positive")
        if self.bar_size not in BAR_AREAS:
            raise ValueError(f"Invalid bar size: {self.bar_size}")
        if self.n_bars < 4:
            raise ValueError("Minimum 4 bars required for circular sections")

    @property
    def Di(self) -> float:
        """Inner diameter of steel tube = concrete core diameter"""
        return self.D_core

    @property
    def Do(self) -> float:
        """Outer diameter of steel tube"""
        return self.Di + 2.0 * self.t_tube

    @property
    def R_conc(self) -> float:
        """Radius of concrete core"""
        return 0.5 * self.D_core

    @property
    def bar_radius(self) -> float:
        """Radius to centerline of reinforcing bars"""
        return max(0.0, self.R_conc - self.cover)

    @property
    def bar_area(self) -> float:
        """Area of single reinforcing bar (in²)"""
        return BAR_AREAS[self.bar_size]

    @property
    def bar_diameter(self) -> float:
        """Nominal diameter of reinforcing bar (in)"""
        return BAR_DIAM[self.bar_size]

# ---------- Mesh Control Dataclass ----------
@dataclass
class MeshCtl:
    n_theta: int = 180      # number of circumferential divisions
    n_rad_conc: int = 36    # number of radial divisions for concrete
    n_ring_steel: int = 3   # number of rings through steel tube thickness

# ---------- Helper Functions ----------
def beta1_aci(fc_ksi: float) -> float:
    """
    ACI 318-19 Table 22.2.2.4.3: Whitney stress block parameter β₁

    β₁ = 0.85                           for f'c ≤ 4 ksi
    β₁ = 0.85 - 0.05(f'c - 4)          for 4 < f'c < 8 ksi
    β₁ = 0.65                           for f'c ≥ 8 ksi
    """
    if fc_ksi <= 4.0:
        return 0.85
    elif fc_ksi >= 8.0:
        return 0.65
    else:
        return max(0.65, min(0.85, 0.85 - 0.05 * (fc_ksi - 4.0)))


def section_props(geom: Geometry) -> Tuple[float, float, float, float, float, float]:
    """
    Calculate section properties

    Returns:
        Ac: Concrete core area (in²)
        As: Steel tube area (in²)
        Ab_tot: Total rebar area (in²)
        Ic: Concrete moment of inertia (in⁴)
        Is: Steel tube moment of inertia (in⁴)
        Ib: Rebar moment of inertia about centroid (in⁴)
    """
    # Concrete core area: A_c = π·D²/4
    Ac = math.pi * (geom.D_core ** 2) / 4.0

    # Steel tube area: A_s = π(D_o² - D_i²)/4
    As = math.pi * (geom.Do ** 2 - geom.Di ** 2) / 4.0

    # Total rebar area
    Ab_one = BAR_AREAS[geom.bar_size]
    Ab_tot = geom.n_bars * Ab_one

    # Concrete moment of inertia: I_c = π·D⁴/64
    Ic = math.pi * (geom.D_core ** 4) / 64.0

    # Steel tube moment of inertia: I_s = π(D_o⁴ - D_i⁴)/64
    Is = math.pi * (geom.Do ** 4 - geom.Di ** 4) / 64.0

    # Rebar moment of inertia (bars on circle at radius r)
    # I_b = n·A_bar·r² (parallel axis theorem, bars treated as point areas)
    r = geom.bar_radius
    Ib = geom.n_bars * Ab_one * (r ** 2)

    return Ac, As, Ab_tot, Ic, Is, Ib


def nice(x: float, nd: int = 3) -> str:
    """Format number with thousands separator and specified decimal places"""
    return f"{x:,.{nd}f}"


def make_output_dir(base: Optional[str] = None) -> Path:
    """Create timestamped output directory"""
    out_base = Path(base or os.getcwd()) / "CFT_Output"
    out_base.mkdir(exist_ok=True)
    run = time.strftime("%Y%m%d_%H%M%S")
    run_dir = out_base / f"run_{run}"
    run_dir.mkdir(exist_ok=True)
    return run_dir


def _safe_csv_write(path: Path, header: List[str], rows: List[List[Any]]) -> Path:
    """Write CSV with fallback for locked files"""
    try:
        with open(path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            if header:
                w.writerow(header)
            w.writerows(rows)
        return path
    except PermissionError:
        alt = path.with_name(f"{path.stem}_{time.strftime('%Y%m%d_%H%M%S')}{path.suffix}")
        with open(alt, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            if header:
                w.writerow(header)
            w.writerows(rows)
        print(f"⚠️ File locked. Wrote to: {alt}")
        return alt


# ---------- Fiber Generation Functions ----------
def gen_concrete_fibers(geom: Geometry, mesh: MeshCtl) -> List[Tuple[float, float, float]]:
    """
    Generate concrete fiber mesh for strain compatibility analysis.

    Divides concrete core into radial rings and circumferential segments.
    Each fiber has coordinates (x, y) and tributary area A.
    """
    R = geom.R_conc
    radii = np.linspace(0.0, R, mesh.n_rad_conc + 1)
    r_mid = 0.5 * (radii[:-1] + radii[1:])
    dr = np.diff(radii)
    dtheta = 2.0 * math.pi / mesh.n_theta

    fibers = []
    for rm, drr in zip(r_mid, dr):
        # Area of each fiber: dA = r·dr·dθ
        A_ring = dtheta * rm * drr
        for j in range(mesh.n_theta):
            th = (j + 0.5) * dtheta
            x = rm * math.cos(th)
            y = rm * math.sin(th)
            fibers.append((x, y, A_ring))

    return fibers


def gen_tube_fibers(geom: Geometry, mesh: MeshCtl) -> List[Tuple[float, float, float]]:
    """
    Generate steel tube fiber mesh.

    Divides tube wall into radial layers and circumferential segments.
    """
    Ri = 0.5 * geom.Di  # Inner radius
    Ro = 0.5 * geom.Do  # Outer radius
    r_layers = np.linspace(Ri, Ro, mesh.n_ring_steel + 1)
    r_mid = 0.5 * (r_layers[:-1] + r_layers[1:])
    dr = np.diff(r_layers)
    dtheta = 2.0 * math.pi / mesh.n_theta

    fibers = []
    for rm, drr in zip(r_mid, dr):
        A_ring = dtheta * rm * drr
        for j in range(mesh.n_theta):
            th = (j + 0.5) * dtheta
            x = rm * math.cos(th)
            y = rm * math.sin(th)
            fibers.append((x, y, A_ring))

    return fibers


def gen_bar_fibers(geom: Geometry) -> List[Tuple[float, float, float]]:
    """
    Generate reinforcing bar fibers.

    Bars are evenly distributed around a circle at radius = bar_radius.
    """
    Abar = BAR_AREAS[geom.bar_size]
    r = geom.bar_radius

    bars = []
    for j in range(geom.n_bars):
        th = 2.0 * math.pi * j / geom.n_bars
        x = r * math.cos(th)
        y = r * math.sin(th)
        bars.append((x, y, Abar))

    return bars


# ---------- Material Stress-Strain Functions ----------
def sig_conc(eps: float, mats: Materials) -> float:
    """
    Concrete stress-strain relationship.

    - Tension: σ = 0 (cracked section assumption)
    - Compression: σ = min(Ec·ε, 0.95·f'c) - linear elastic up to cap

    Note: Positive strain = compression (fiber analysis convention)
    """
    if eps >= 0.0:
        # Compression - linear elastic capped at 0.95·f'c for round CFT
        return min(mats.Ec * eps, mats.conf_cap * mats.fc)
    else:
        # Tension - concrete cracks, no tensile capacity
        return 0.0


def sig_steel(eps: float, Fy: float, Es: float) -> float:
    """
    Steel stress-strain relationship (elastic-perfectly plastic).

    σ = Es·ε,  clipped to ±Fy
    """
    s = Es * eps
    return max(-Fy, min(Fy, s))


def strain_from_c(y: float, R: float, c: float, eps_cu: float) -> float:
    """
    Calculate strain at fiber location y given neutral axis depth c.

    Strain profile: ε(y) = εcu · (y - y_NA) / c
    where y_NA = R - c (neutral axis measured from bottom)

    Convention:
    - y = +R at top (compression face, ε = +εcu)
    - y = -R at bottom (tension face)
    - Positive strain = compression
    """
    y_NA = R - c
    return eps_cu * (y - y_NA) / max(c, 1e-12)


# ---------- Section Force Integration ----------
def fiber_forces_moment_from_c(
    c: float,
    conc_f: List[Tuple[float, float, float]],
    tube_f: List[Tuple[float, float, float]],
    bar_f: List[Tuple[float, float, float]],
    mats: Materials,
    R: float
) -> Tuple[float, float, float, float]:
    """
    Integrate fiber stresses to get section forces and moment.

    Returns:
        T: Total tensile force (kip)
        Cc: Total compressive force (kip)
        Pn: Net axial force = Cc - T (kip, positive = compression)
        Mn: Moment about centroid (kip-in)
    """
    T = 0.0
    Cc = 0.0
    Mx = 0.0

    # Concrete fibers
    for x, y, A in conc_f:
        eps = strain_from_c(y, R, c, mats.eps_cu)
        s = sig_conc(eps, mats)
        F = s * A
        if F >= 0:
            Cc += F
        else:
            T += -F
        Mx += F * y

    # Steel tube fibers
    for x, y, A in tube_f:
        eps = strain_from_c(y, R, c, mats.eps_cu)
        s = sig_steel(eps, mats.Fy_tube, mats.Es)
        F = s * A
        if F >= 0:
            Cc += F
        else:
            T += -F
        Mx += F * y

    # Reinforcing bar fibers
    for x, y, A in bar_f:
        eps = strain_from_c(y, R, c, mats.eps_cu)
        s = sig_steel(eps, mats.fy_bar, mats.Es)
        F = s * A
        if F >= 0:
            Cc += F
        else:
            T += -F
        Mx += F * y

    Pn = Cc - T
    return T, Cc, Pn, abs(Mx)


def pure_compression_plastic(
    conc_f: List[Tuple[float, float, float]],
    tube_f: List[Tuple[float, float, float]],
    bar_f: List[Tuple[float, float, float]],
    mats: Materials
) -> Tuple[float, float, float, float]:
    """
    Calculate pure compression capacity (all materials at yield/ultimate).

    P_n = 0.95·f'c·A_c + F_y,tube·A_s + f_y,bar·A_b
    """
    T = 0.0
    Cc = 0.0
    Mx = 0.0

    for x, y, A in conc_f:
        F = mats.conf_cap * mats.fc * A
        Cc += F
        Mx += F * y

    for x, y, A in tube_f:
        F = mats.Fy_tube * A
        Cc += F
        Mx += F * y

    for x, y, A in bar_f:
        F = mats.fy_bar * A
        Cc += F
        Mx += F * y

    Pn = Cc
    return T, Cc, Pn, abs(Mx)


def solve_c_for_pure_bending(
    conc_f: List[Tuple[float, float, float]],
    tube_f: List[Tuple[float, float, float]],
    bar_f: List[Tuple[float, float, float]],
    mats: Materials,
    R: float,
    c_lo: float,
    c_hi: float,
    tol: float = 1e-6,
    itmax: int = 80
) -> float:
    """
    Find neutral axis depth c for pure bending (Pn = 0) using bisection.
    """
    def Pn_of(c):
        return fiber_forces_moment_from_c(c, conc_f, tube_f, bar_f, mats, R)[2]

    f_lo = Pn_of(c_lo)
    f_hi = Pn_of(c_hi)

    # Expand bounds if needed to bracket root
    tries = 0
    while f_lo * f_hi > 0 and tries < 20:
        c_lo *= 0.8
        c_hi = min(2.0 * R, c_hi * 1.05)
        f_lo = Pn_of(c_lo)
        f_hi = Pn_of(c_hi)
        tries += 1

    if f_lo * f_hi > 0:
        return c_lo if abs(f_lo) < abs(f_hi) else c_hi

    # Bisection iteration
    for _ in range(itmax):
        cm = 0.5 * (c_lo + c_hi)
        fm = Pn_of(cm)
        if abs(fm) < tol or abs(c_hi - c_lo) < 1e-6:
            return cm
        if f_lo * fm <= 0:
            c_hi, f_hi = cm, fm
        else:
            c_lo, f_lo = cm, fm

    return 0.5 * (c_lo + c_hi)


# ---------- ACI 318 Table 21.2.2 φ-Factors ----------
PHI_FLEX_TENSION = 0.90       # tension-controlled / flexure
PHI_COMP_TIED = 0.65          # compression-controlled (tied)
PHI_COMP_SPIRAL = 0.75        # compression-controlled (spiral)
EPS_T_MIN = 0.002             # compression-controlled limit
EPS_T_MAX = 0.005             # tension-controlled limit


def phi_aci_from_eps_t(eps_t: float, Pn: float, spiral: bool = False) -> float:
    """
    Calculate strength reduction factor φ per ACI 318-19 Table 21.2.2.

    Parameters:
        eps_t: Net tensile strain in extreme tension steel
        Pn: Nominal axial force (positive = compression)
        spiral: True for spiral reinforcement, False for tied

    Returns:
        φ: Strength reduction factor
    """
    if Pn < 0.0:
        # Net tension - use flexure/tension φ
        return PHI_FLEX_TENSION

    phi_comp = PHI_COMP_SPIRAL if spiral else PHI_COMP_TIED

    if eps_t >= EPS_T_MAX:
        return PHI_FLEX_TENSION
    if eps_t <= EPS_T_MIN:
        return phi_comp

    # Linear interpolation in transition zone
    w = (eps_t - EPS_T_MIN) / (EPS_T_MAX - EPS_T_MIN)
    return phi_comp + w * (PHI_FLEX_TENSION - phi_comp)


# ---------- Shear Capacity Functions ----------
def shear_strengths_aci_cft(
    geom: Geometry,
    mats: Materials,
    steel_phi: float = 0.90,
    conc_phi: float = 0.75
) -> Dict[str, float]:
    """
    Calculate shear capacities per ACI 318-19 and AISC 360-22.

    Concrete (ACI 318-19 §22.5.5.1):
        V_c = 2·λ·√(f'c[psi])·b_w·d / 1000 [kip]
        where d = 0.8·D (effective depth for circular sections)
              b_w = D (width taken as diameter)

    Steel tube (AISC 360-22 §G2.1):
        V_n,steel = 0.6·F_y·A_v
        where A_v ≈ 2·A_s/π (web area for round HSS)

    Returns:
        Dictionary with nominal and factored shear capacities
    """
    D = geom.D_core
    d_eff = 0.8 * D  # Effective depth (ACI 318-19 §22.5.2.1 for circular)
    bw = D           # Width taken as diameter

    # Concrete Vc (ACI 318-19 Eq. 22.5.5.1)
    fc_psi = mats.fc * 1000.0
    Vc_nom = 2.0 * mats.lambda_nw * math.sqrt(fc_psi) * bw * d_eff / 1000.0  # kip

    # Steel tube shear
    Ac, As, Ab, Ic, Is, Ib = section_props(geom)
    Av = 2.0 * As / math.pi  # Effective shear area for round HSS
    Vn_steel_nom = 0.6 * mats.Fy_tube * Av  # kip

    # Strength-reduced values
    phiVc = conc_phi * Vc_nom
    phiV_steel = steel_phi * Vn_steel_nom
    Vn_sum_nom = Vc_nom + Vn_steel_nom
    phiV_sum = phiVc + phiV_steel

    return {
        'D': D,
        'd': d_eff,
        'bw': bw,
        'Vc_nom': Vc_nom,
        'phiVc': phiVc,
        'Vn_steel_nom': Vn_steel_nom,
        'phiV_steel': phiV_steel,
        'Vn_sum_nom': Vn_sum_nom,
        'phiV_sum': phiV_sum,
        'Av': Av,
        'As': As
    }


# ---------- P-M Envelope Generation ----------
def build_PM_cloud(
    geom: Geometry,
    mats: Materials,
    mesh: MeshCtl,
    n_c: int = 700,
    return_eps: bool = False
) -> Tuple[np.ndarray, ...]:
    """
    Build P-M interaction diagram by sweeping neutral axis depth c.

    Method:
        1. Fix extreme compression fiber strain at εcu = 0.003
        2. Vary c from very small (pure tension) to very large (pure compression)
        3. For each c, integrate fiber stresses to get Pn, Mn
        4. Mirror about M = 0 for complete envelope

    Returns:
        Pn_cloud, Mn_cloud [, Et_cloud if return_eps]
    """
    conc = gen_concrete_fibers(geom, mesh)
    tube = gen_tube_fibers(geom, mesh)
    bars = gen_bar_fibers(geom)
    R = 0.5 * geom.Do  # Use outer radius for strain profile

    P_list = []
    M_list = []
    E_list = []

    # Pure compression point (all materials at yield)
    _, _, Pn_pc, Mn_pc = pure_compression_plastic(conc, tube, bars, mats)
    P_list.append(Pn_pc)
    M_list.append(Mn_pc)
    E_list.append(0.0)

    # Sweep c from small to large
    c_vals = np.linspace(1e-5, 2.0 * R * (1 - 1e-6), n_c)
    for c in c_vals:
        _, _, Pn, Mn = fiber_forces_moment_from_c(c, conc, tube, bars, mats, R)
        eps_t = mats.eps_cu * (2.0 * R - c) / max(c, 1e-12)
        P_list.append(Pn)
        M_list.append(Mn)
        E_list.append(eps_t)

    P = np.array(P_list)
    M = np.array(M_list)
    Et = np.array(E_list)

    # Mirror about M = 0 (symmetric section)
    P_sym = np.concatenate([P, P])
    M_sym = np.concatenate([M, -M])
    Et_sym = np.concatenate([Et, Et])

    # Sort by moment for proper envelope
    idx = np.argsort(M_sym)

    if return_eps:
        return P_sym[idx], M_sym[idx], Et_sym[idx]
    return P_sym[idx], M_sym[idx]


def convex_hull(points: np.ndarray) -> np.ndarray:
    """
    Compute convex hull using Andrew's monotone chain algorithm.
    """
    pts = sorted(set(map(tuple, points)))
    if len(pts) <= 1:
        return np.array(pts)

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    # Build lower hull
    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    # Build upper hull
    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    hull = lower[:-1] + upper[:-1]
    return np.array(hull)


# ---------- Key Control Points ----------
def key_points_table(geom: Geometry, mats: Materials, mesh: MeshCtl) -> List[Dict]:
    """
    Calculate key control points on P-M diagram.
    """
    conc = gen_concrete_fibers(geom, mesh)
    tube = gen_tube_fibers(geom, mesh)
    bars = gen_bar_fibers(geom)
    R = 0.5 * geom.Do
    beta1 = beta1_aci(mats.fc)

    def et_from_c(c):
        return mats.eps_cu * (2.0 * R - c) / max(c, 1e-12)

    rows = []

    # Point 1: Pure compression
    _, _, Pn, Mn = pure_compression_plastic(conc, tube, bars, mats)
    rows.append({
        'Definition': "Pure compression (conc.@0.95f'c; steel @Fy)",
        'c': None, 'a': None, 'eps_t': 0.0, 'f_t': None,
        'Pn': Pn, 'Mn': Mn
    })

    # Point 2: εt = 0 at tension face (c ≈ 2R)
    c_ept0 = 2.0 * R
    _, _, Pn, Mn = fiber_forces_moment_from_c(c_ept0, conc, tube, bars, mats, R)
    rows.append({
        'Definition': "ε_t = 0 at tension face (c ≈ 2R)",
        'c': c_ept0, 'a': beta1 * c_ept0, 'eps_t': 0.0, 'f_t': 0.0,
        'Pn': Pn, 'Mn': Mn
    })

    # Point 3: Balanced condition (c = R)
    c_bal = R
    et_bal = et_from_c(c_bal)
    _, _, Pn, Mn = fiber_forces_moment_from_c(c_bal, conc, tube, bars, mats, R)
    rows.append({
        'Definition': "Balance (ε_cu = ε_t) (c = R)",
        'c': c_bal, 'a': beta1 * c_bal, 'eps_t': et_bal, 'f_t': None,
        'Pn': Pn, 'Mn': Mn
    })

    # Point 4: Yield strain at tension face (εt = εy)
    eps_y = mats.fy_bar / mats.Es
    c_y = 2.0 * R / (1.0 + eps_y / max(mats.eps_cu, 1e-12))
    _, _, Pn, Mn = fiber_forces_moment_from_c(c_y, conc, tube, bars, mats, R)
    rows.append({
        'Definition': "Yield strain at tension face (ε_t = ε_y)",
        'c': c_y, 'a': beta1 * c_y, 'eps_t': eps_y, 'f_t': mats.fy_bar,
        'Pn': Pn, 'Mn': Mn
    })

    # Point 5: Pure bending (Pn = 0)
    c_pb = solve_c_for_pure_bending(conc, tube, bars, mats, R,
                                     c_lo=1e-6 * R, c_hi=2.0 * R * (1 - 1e-6))
    et_pb = et_from_c(c_pb)
    _, _, Pn, Mn = fiber_forces_moment_from_c(c_pb, conc, tube, bars, mats, R)
    rows.append({
        'Definition': "Pure bending (P_n = 0)",
        'c': c_pb, 'a': beta1 * c_pb, 'eps_t': et_pb,
        'f_t': min(mats.fy_bar, mats.Es * et_pb),
        'Pn': Pn, 'Mn': Mn
    })

    # Point 6: Pure tension
    c_tiny = 1e-6 * R
    et_pt = et_from_c(c_tiny)
    _, _, Pn, Mn = fiber_forces_moment_from_c(c_tiny, conc, tube, bars, mats, R)
    rows.append({
        'Definition': "Pure tension (steel only; conc. cracked)",
        'c': 0.0, 'a': 0.0, 'eps_t': et_pt,
        'f_t': min(mats.fy_bar, mats.Es * et_pt),
        'Pn': Pn, 'Mn': Mn
    })

    # Add point numbers
    for i, r in enumerate(rows, start=1):
        r['Pt'] = i

    return rows


def pick_peak_moment_point(P: np.ndarray, M: np.ndarray) -> Tuple[float, float]:
    """Find point with maximum absolute moment."""
    i = int(np.argmax(np.abs(M)))
    return P[i], M[i]


# ---------- CSV Export Functions ----------
def save_key_points_csv(rows: List[Dict], peak_point: Tuple[float, float], out_dir: Path):
    """Save key points to CSV."""
    header = ["Pt", "Definition", "c (in)", "a=β₁c (in)", "ε_t", "f_t (ksi)",
              "P_n (kip)", "M_n (kip-ft)", "M_n (kip-in)"]
    data = []
    for r in rows:
        data.append([
            r['Pt'], r['Definition'],
            "" if r['c'] is None else f"{r['c']:.5f}",
            "" if r['a'] is None else f"{r['a']:.5f}",
            f"{r['eps_t']:.5f}",
            "" if r['f_t'] is None else f"{r['f_t']:.2f}",
            f"{r['Pn']:.2f}",
            f"{r['Mn']/12.0:.2f}",
            f"{r['Mn']:.2f}"
        ])

    Pp, Mp = peak_point
    data += [[""], ["–", "Peak moment from envelope", "", "", "", "",
                    f"{Pp:.2f}", f"{Mp/12.0:.2f}", f"{Mp:.2f}"]]

    path = out_dir / "Key_Points.csv"
    final_path = _safe_csv_write(path, header, data)
    print(f"✓ Saved key points → {final_path}")


def save_PM_points_csv(Pn: np.ndarray, Mn: np.ndarray, out_dir: Path,
                       fname: str = "PM_points.csv"):
    """Save P-M cloud points to CSV."""
    header = ["M_n (kip-ft)", "P_n (kip)", "M_n (kip-in)"]
    data = [[M/12.0, P, M] for P, M in zip(Pn, Mn)]
    path = out_dir / fname
    final_path = _safe_csv_write(path, header, data)
    print(f"✓ Saved PM points → {final_path}")


# ---------- Factored Envelope Calculation ----------
def factored_points_from_nominal(
    Pn: np.ndarray,
    Mn: np.ndarray,
    Et: np.ndarray,
    spiral: bool = False
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Apply ACI 318-19 Table 21.2.2 φ factors to nominal values."""
    phi = np.array([phi_aci_from_eps_t(et, p, spiral=spiral)
                    for et, p in zip(Et, Pn)], dtype=float)
    return phi * Pn, phi * Mn, phi


# ---------- Plotting Functions ----------
def plot_section(geom: Geometry, out_png: str):
    """Generate section sketch showing tube, concrete, and rebars."""
    th = np.linspace(0, 2 * np.pi, 600)

    # Outer tube
    x_out = (geom.Do / 2) * np.cos(th)
    y_out = (geom.Do / 2) * np.sin(th)

    # Inner tube / concrete core boundary
    x_in = (geom.Di / 2) * np.cos(th)
    y_in = (geom.Di / 2) * np.sin(th)

    fig, ax = plt.subplots(figsize=(8, 8))

    # Fill concrete core
    ax.fill(x_in, y_in, color='lightgray', alpha=0.5, label='Concrete core')

    # Steel tube
    ax.plot(x_out, y_out, 'b-', lw=2, label=f'Steel tube (Do={geom.Do:.2f}")')
    ax.plot(x_in, y_in, 'b--', lw=1.5, label=f'Tube inner (Di={geom.Di:.2f}")')

    # Rebars
    r = geom.bar_radius
    bx = [r * math.cos(2 * np.pi * i / geom.n_bars) for i in range(geom.n_bars)]
    by = [r * math.sin(2 * np.pi * i / geom.n_bars) for i in range(geom.n_bars)]
    ax.scatter(bx, by, s=80, c='red', marker='o', zorder=5,
               label=f'{geom.n_bars}-#{geom.bar_size} @ r={r:.2f}"')

    # Bar circle
    x_bar = r * np.cos(th)
    y_bar = r * np.sin(th)
    ax.plot(x_bar, y_bar, 'r:', lw=1, alpha=0.7)

    # Dimensions
    ax.annotate('', xy=(geom.Do/2, 0), xytext=(-geom.Do/2, 0),
                arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
    ax.text(0, -geom.Do/2 - 2, f'D_o = {geom.Do:.3f}"', ha='center', fontsize=10)

    ax.set_aspect('equal')
    ax.set_xlabel('x (in)')
    ax.set_ylabel('y (in)')
    ax.set_title('Round CFT Section', fontsize=14, fontweight='bold')
    ax.grid(True, lw=0.3, alpha=0.5)
    ax.legend(loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.savefig(out_png, dpi=240)
    plt.close()


def plot_PM_envelope(
    Pn: np.ndarray,
    Mn: np.ndarray,
    Et: np.ndarray,
    out_png: str,
    add_cloud: bool = True,
    spiral: bool = False
):
    """
    Plot P-M interaction diagram with nominal and factored envelopes.
    """
    # Nominal hull
    pts_nom = np.column_stack([Mn / 12.0, Pn])
    hull_nom = convex_hull(pts_nom)

    # Factored per-point + hull
    Pf, Mf, phi_arr = factored_points_from_nominal(Pn, Mn, Et, spiral=spiral)
    pts_fac = np.column_stack([Mf / 12.0, Pf])
    hull_fac = convex_hull(pts_fac)

    fig, ax = plt.subplots(figsize=(10, 8))

    if add_cloud:
        ax.scatter(pts_nom[:, 0], pts_nom[:, 1], s=6, alpha=0.22,
                   c='blue', label='c-sweep cloud (nominal)')

    # Nominal envelope
    ax.plot(hull_nom[:, 0], hull_nom[:, 1], 'b-', lw=2.0,
            label='Nominal envelope (Pn, Mn)')

    # Factored envelope
    ax.plot(hull_fac[:, 0], hull_fac[:, 1], 'r-', lw=2.0,
            label='Factored envelope (φPn, φMn) per ACI 318-19 §21.2.2')

    # 0.8 cutoff lines
    x_min = float(np.min(pts_nom[:, 0]) * 1.05)
    x_max = float(np.max(pts_nom[:, 0]) * 1.05)
    x_span = [x_min, x_max]

    y80_nom = 0.80 * float(np.max(Pn))
    ax.plot(x_span, [y80_nom, y80_nom], 'b--', lw=1.2,
            label=f'0.8·Pn,max = {y80_nom:,.0f} kip')

    y80_fac = 0.80 * float(np.max(Pf))
    ax.plot(x_span, [y80_fac, y80_fac], 'r--', lw=1.2,
            label=f'0.8·φPn,max = {y80_fac:,.0f} kip')

    ax.set_xlabel('Mn (kip-ft)', fontsize=12)
    ax.set_ylabel('Pn (kip)', fontsize=12)
    ax.set_title('P–M Interaction Diagram\nNominal vs. Factored (ACI 318-19)',
                 fontsize=14, fontweight='bold')
    ax.grid(True, lw=0.3, alpha=0.5)
    ax.legend(loc='upper right', fontsize=9)

    plt.tight_layout()
    plt.savefig(out_png, dpi=260)
    plt.close()


# ---------- PDF Report Generation ----------
class NumberedCanvas(canvas.Canvas):
    """Canvas with page numbering."""
    def __init__(self, *args, **kwargs):
        canvas.Canvas.__init__(self, *args, **kwargs)
        self._saved_page_states = []

    def showPage(self):
        self._saved_page_states.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        num_pages = len(self._saved_page_states)
        for state in self._saved_page_states:
            self.__dict__.update(state)
            self.draw_page_number(num_pages)
            canvas.Canvas.showPage(self)
        canvas.Canvas.save(self)

    def draw_page_number(self, page_count):
        self.setFont("Helvetica", 9)
        self.drawRightString(
            LETTER[0] - 36,
            28,
            f"Page {self._pageNumber} of {page_count}"
        )


def scale_logo(logo_path: str, max_width: float, max_height: float) -> Tuple[float, float]:
    """
    Calculate scaled dimensions for logo to fit within bounds.

    Returns:
        (width, height) in points
    """
    try:
        with PILImage.open(logo_path) as img:
            orig_width, orig_height = img.size

            # Calculate scale factors
            scale_w = max_width / orig_width
            scale_h = max_height / orig_height
            scale = min(scale_w, scale_h, 1.0)  # Don't upscale

            return orig_width * scale, orig_height * scale
    except Exception as e:
        print(f"⚠️ Could not read logo: {e}")
        return 0, 0


def build_title_block(
    project: ProjectInfo,
    page_width: float,
    styles: Dict
) -> Table:
    """
    Create professional title block with optional logo.

    Logo is dynamically scaled to fit within allocated space.
    """
    # Title block dimensions
    logo_max_width = 1.2 * inch
    logo_max_height = 0.8 * inch

    # Prepare logo element
    logo_elem = ""
    if project.logo_path and os.path.exists(project.logo_path):
        w, h = scale_logo(project.logo_path, logo_max_width, logo_max_height)
        if w > 0 and h > 0:
            logo_elem = Image(project.logo_path, width=w, height=h)

    # Title block content
    title_text = Paragraph(
        f"<b>{project.project_name}</b>",
        styles['TitleBlock']
    )

    info_left = Paragraph(
        f"<b>Project:</b> {project.project_number}<br/>"
        f"<b>Client:</b> {project.client}<br/>"
        f"<b>Location:</b> {project.location}",
        styles['Info']
    )

    info_right = Paragraph(
        f"<b>Engineer:</b> {project.engineer}<br/>"
        f"<b>Checker:</b> {project.checker}<br/>"
        f"<b>Date:</b> {project.date} | <b>Rev:</b> {project.revision}",
        styles['Info']
    )

    # Build table layout
    available_width = page_width - 72  # 36pt margins on each side
    col_widths = [1.3 * inch, 3.0 * inch, 2.6 * inch]

    # Adjust for available width
    total = sum(col_widths)
    scale = available_width / total
    col_widths = [w * scale for w in col_widths]

    tbl_data = [[logo_elem, title_text, ""]]
    tbl_data.append([info_left, "", info_right])

    tbl = Table(tbl_data, colWidths=col_widths, rowHeights=[0.9 * inch, 0.6 * inch])
    tbl.setStyle(TableStyle([
        ('SPAN', (1, 0), (2, 0)),  # Title spans columns
        ('SPAN', (0, 1), (1, 1)),  # Left info spans
        ('ALIGN', (0, 0), (0, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('ALIGN', (1, 0), (1, 0), 'CENTER'),
        ('ALIGN', (2, 1), (2, 1), 'RIGHT'),
        ('BOX', (0, 0), (-1, -1), 1.5, colors.black),
        ('INNERGRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ('BACKGROUND', (0, 0), (-1, 0), colors.Color(0.95, 0.95, 0.98)),
        ('LEFTPADDING', (0, 0), (-1, -1), 6),
        ('RIGHTPADDING', (0, 0), (-1, -1), 6),
        ('TOPPADDING', (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
    ]))

    return tbl


def build_pdf(
    out_pdf_path: str,
    geom: Geometry,
    mats: Materials,
    project: ProjectInfo,
    phi_c: float,
    phi_b: float,
    phi_v_steel: float,
    fig_env_path: str,
    fig_section_path: str,
    key_rows: List[Dict],
    axial_phiPn: float,
    shear_info: Dict,
    Mn_peak_nom: float,
    phiMn_peak_nom: float,
    Mn_peak_fac: float
):
    """
    Generate comprehensive PDF report with title block on every page and highlighted results.
    ALL SUBSCRIPTS/SUPERSCRIPTS PROPERLY FORMATTED FOR TABLES AND TEXT.
    """
    # Page setup
    page_width, page_height = LETTER

    # Create styles
    styles = getSampleStyleSheet()

    # Custom styles - ALL BLACK TEXT
    styles.add(ParagraphStyle(
        name='TitleBlock',
        parent=styles['Title'],
        fontSize=16,
        leading=20,
        alignment=TA_CENTER,
        textColor=colors.black
    ))

    styles.add(ParagraphStyle(
        name='Info',
        parent=styles['Normal'],
        fontSize=9,
        leading=12,
        textColor=colors.black
    ))

    styles.add(ParagraphStyle(
        name='H1',
        parent=styles['Title'],
        fontSize=14,
        leading=17,
        spaceAfter=10,
        spaceBefore=10,
        textColor=colors.black,
        fontName='Helvetica-Bold'
    ))

    styles.add(ParagraphStyle(
        name='H2',
        parent=styles['Heading2'],
        fontSize=11,
        leading=14,
        spaceAfter=6,
        spaceBefore=8,
        textColor=colors.black,
        fontName='Helvetica-Bold'
    ))

    styles.add(ParagraphStyle(
        name='H3',
        parent=styles['Heading3'],
        fontSize=10,
        leading=12,
        spaceAfter=4,
        spaceBefore=6,
        fontName='Helvetica-Bold',
        textColor=colors.black
    ))

    styles.add(ParagraphStyle(
        name='Equation',
        fontName='Courier',
        fontSize=9,
        leading=12,
        leftIndent=20,
        spaceBefore=3,
        spaceAfter=3,
        textColor=colors.black
    ))

    styles.add(ParagraphStyle(
        name='Result',
        fontName='Courier-Bold',
        fontSize=9,
        leading=12,
        leftIndent=20,
        spaceBefore=2,
        spaceAfter=4,
        textColor=colors.black
    ))

    styles.add(ParagraphStyle(
        name='HighlightResult',
        fontName='Helvetica-Bold',
        fontSize=11,
        leading=14,
        leftIndent=20,
        spaceBefore=4,
        spaceAfter=6,
        textColor=colors.black
    ))

    styles.add(ParagraphStyle(
        name='Caption',
        parent=styles['Normal'],
        fontSize=9,
        alignment=TA_CENTER,
        spaceBefore=4,
        spaceAfter=8,
        textColor=colors.black,
        fontName='Helvetica-Oblique'
    ))

    styles.add(ParagraphStyle(
        name='TableCell',
        parent=styles['Normal'],
        fontSize=9,
        leading=11,
        textColor=colors.black
    ))

    # Update existing Normal style instead of adding new one
    styles['Normal'].fontSize = 10
    styles['Normal'].leading = 12
    styles['Normal'].spaceAfter = 4
    styles['Normal'].textColor = colors.black

    # Helper function to wrap table cells in Paragraphs for HTML rendering
    def wrap_cell(text, style_name='TableCell'):
        """Wrap table cell text in Paragraph for HTML tag support"""
        if isinstance(text, str):
            return Paragraph(text, styles[style_name])
        return text

    # Custom canvas for header on every page
    class HeaderCanvas(canvas.Canvas):
        def __init__(self, *args, **kwargs):
            canvas.Canvas.__init__(self, *args, **kwargs)
            self._saved_page_states = []
            self.project_info = project

        def showPage(self):
            self._saved_page_states.append(dict(self.__dict__))
            self._startPage()

        def save(self):
            num_pages = len(self._saved_page_states)
            for state in self._saved_page_states:
                self.__dict__.update(state)
                self.draw_header()
                self.draw_page_number(num_pages)
                canvas.Canvas.showPage(self)
            canvas.Canvas.save(self)

        def draw_header(self):
            """Draw title block header on every page"""
            if self._pageNumber == 1:
                return  # First page has full title block

            # Draw simple header bar
            self.setFillColor(colors.Color(0.85, 0.85, 0.85))
            self.rect(36, page_height - 60, page_width - 72, 24, fill=1, stroke=0)

            # Project name
            self.setFillColor(colors.black)
            self.setFont("Helvetica-Bold", 11)
            self.drawString(42, page_height - 48, self.project_info.project_name)

            # Project number and date
            self.setFont("Helvetica", 9)
            self.drawRightString(page_width - 42, page_height - 48,
                               f"Project: {self.project_info.project_number} | {self.project_info.date}")

        def draw_page_number(self, page_count):
            self.setFillColor(colors.black)
            self.setFont("Helvetica", 9)
            self.drawRightString(
                page_width - 36,
                28,
                f"Page {self._pageNumber} of {page_count}"
            )

    # Document with custom canvas
    doc = SimpleDocTemplate(
        out_pdf_path,
        pagesize=LETTER,
        leftMargin=36,
        rightMargin=36,
        topMargin=70,  # Extra space for header
        bottomMargin=36
    )

    # Calculate section properties
    Ac, As, Ab, Ic, Is, Ib = section_props(geom)
    beta1 = beta1_aci(mats.fc)

    # Flow content
    flow = []

    # ========== TITLE BLOCK (PAGE 1 ONLY) ==========
    title_tbl = build_title_block(project, page_width, styles)
    flow.append(title_tbl)
    flow.append(Spacer(1, 10))

    # ========== REPORT TITLE ==========
    flow.append(Paragraph("<b>ENCASED DRILLED PIER ANALYSIS</b>", styles['H1']))
    flow.append(Paragraph("P-M Interaction &amp; Capacity Assessment per ACI 318-19 &amp; AISC 360-22", styles['Normal']))
    flow.append(Spacer(1, 8))
    flow.append(HRFlowable(width="100%", thickness=1.5, color=colors.black, spaceAfter=10))

    # ========== SECTION DIAGRAM (FIRST!) ==========
    flow.append(Paragraph("<b>SECTION DIAGRAM</b>", styles['H2']))

    if os.path.exists(fig_section_path):
        img_width = 4.5 * inch
        img = Image(fig_section_path, width=img_width, height=img_width)
        flow.append(img)
        flow.append(Paragraph("Figure 1: Cross-section showing steel tube, concrete core, and reinforcing bars",
                             styles['Caption']))
    flow.append(Spacer(1, 10))

    # ========== P-M INTERACTION DIAGRAM (SECOND!) ==========
    flow.append(Paragraph("<b>P-M INTERACTION DIAGRAM</b>", styles['H2']))

    if os.path.exists(fig_env_path):
        img_width = 6.5 * inch
        img_height = 5.2 * inch
        img = Image(fig_env_path, width=img_width, height=img_height)
        flow.append(img)
        flow.append(Paragraph("Figure 2: P-M interaction diagram showing nominal and factored envelopes",
                             styles['Caption']))

    # ========== FACTORED DESIGN CAPACITIES SUMMARY (HIGHLIGHTED!) ==========
    flow.append(Spacer(1, 12))
    flow.append(HRFlowable(width="100%", thickness=2, color=colors.black, spaceBefore=6, spaceAfter=6))
    flow.append(Paragraph("<b>FACTORED DESIGN CAPACITIES SUMMARY</b>", styles['H1']))

    # Create highlighted summary table - WRAP ALL CELLS
    summary_data = [
        [wrap_cell("<b>Capacity Type</b>"), wrap_cell("<b>Factored Design Value</b>")],
        [wrap_cell("Axial Compression (φP<sub>n</sub>)"), wrap_cell(f"{axial_phiPn:,.1f} kip")],
        [wrap_cell("Axial (0.80·φP<sub>n</sub> limit)"), wrap_cell(f"{0.80*axial_phiPn:,.1f} kip")],
        [wrap_cell("Peak Moment (φM<sub>n,max</sub>)"), wrap_cell(f"{Mn_peak_fac/12.0:,.1f} kip-ft")],
        [wrap_cell("Concrete Shear (φV<sub>c</sub>)"), wrap_cell(f"{shear_info['phiVc']:,.1f} kip")],
        [wrap_cell("Steel Shear (φV<sub>steel</sub>)"), wrap_cell(f"{shear_info['phiV_steel']:,.1f} kip")],
        [wrap_cell("Total Shear (φV<sub>total</sub>)"), wrap_cell(f"{shear_info['phiV_sum']:,.1f} kip")],
    ]

    summary_tbl = Table(summary_data, colWidths=[3.5*inch, 2.0*inch])
    summary_tbl.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.black),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 11),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 10),
        ('TOPPADDING', (0, 0), (-1, 0), 10),
        ('BACKGROUND', (0, 1), (-1, -1), colors.Color(1.0, 1.0, 0.7)),  # Light yellow highlight
        ('GRID', (0, 0), (-1, -1), 1.5, colors.black),
        ('FONTNAME', (0, 1), (0, -1), 'Helvetica-Bold'),
        ('FONTNAME', (1, 1), (1, -1), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 1), (-1, -1), 11),
        ('ALIGN', (1, 1), (1, -1), 'RIGHT'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('LEFTPADDING', (0, 0), (-1, -1), 10),
        ('RIGHTPADDING', (0, 0), (-1, -1), 10),
        ('TOPPADDING', (0, 1), (-1, -1), 8),
        ('BOTTOMPADDING', (0, 1), (-1, -1), 8),
    ]))
    flow.append(summary_tbl)
    flow.append(HRFlowable(width="100%", thickness=2, color=colors.black, spaceBefore=6, spaceAfter=12))

    # ========== PAGE BREAK BEFORE DETAILED CALCULATIONS ==========
    flow.append(PageBreak())

    # ========== 1. GEOMETRY & MATERIALS ==========
    flow.append(Paragraph("<b>1. GEOMETRY &amp; MATERIAL PROPERTIES</b>", styles['H2']))

    # Geometry table - WRAP ALL CELLS
    geom_data = [
        [wrap_cell("<b>Geometry</b>"), wrap_cell("<b>Value</b>"), wrap_cell("<b>Unit</b>")],
        [wrap_cell("Concrete core diameter (D<sub>core</sub>)"), wrap_cell(f"{geom.D_core:.3f}"), wrap_cell("in")],
        [wrap_cell("Steel tube wall thickness (t)"), wrap_cell(f"{geom.t_tube:.3f}"), wrap_cell("in")],
        [wrap_cell("Outer diameter (D<sub>o</sub>)"), wrap_cell(f"{geom.Do:.3f}"), wrap_cell("in")],
        [wrap_cell("Inner diameter (D<sub>i</sub>)"), wrap_cell(f"{geom.Di:.3f}"), wrap_cell("in")],
        [wrap_cell("Clear cover to rebar CL"), wrap_cell(f"{geom.cover:.3f}"), wrap_cell("in")],
        [wrap_cell("Number of rebars"), wrap_cell(f"{geom.n_bars}"), wrap_cell("—")],
        [wrap_cell("Rebar size"), wrap_cell(f"#{geom.bar_size}"), wrap_cell("—")],
        [wrap_cell("Rebar diameter"), wrap_cell(f"{geom.bar_diameter:.3f}"), wrap_cell("in")],
        [wrap_cell("Rebar area (each)"), wrap_cell(f"{geom.bar_area:.3f}"), wrap_cell("in<super>2</super>")],
        [wrap_cell("Rebar circle radius"), wrap_cell(f"{geom.bar_radius:.3f}"), wrap_cell("in")],
    ]

    geom_tbl = Table(geom_data, colWidths=[3.2*inch, 1.3*inch, 0.7*inch])
    geom_tbl.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.Color(0.7, 0.7, 0.7)),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 9),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 8),
        ('BACKGROUND', (0, 1), (-1, -1), colors.Color(0.95, 0.95, 0.95)),
        ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ('ALIGN', (1, 1), (1, -1), 'RIGHT'),
        ('ALIGN', (2, 0), (2, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('LEFTPADDING', (0, 0), (-1, -1), 6),
        ('RIGHTPADDING', (0, 0), (-1, -1), 6),
        ('TOPPADDING', (0, 0), (-1, -1), 5),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
    ]))
    flow.append(geom_tbl)
    flow.append(Spacer(1, 8))

    # Materials table - WRAP ALL CELLS
    mat_data = [
        [wrap_cell("<b>Material Property</b>"), wrap_cell("<b>Value</b>"), wrap_cell("<b>Unit</b>")],
        [wrap_cell("Concrete strength (f'<sub>c</sub>)"), wrap_cell(f"{mats.fc:.2f}"), wrap_cell("ksi")],
        [wrap_cell("Concrete modulus (E<sub>c</sub>)"), wrap_cell(f"{mats.Ec:,.0f}"), wrap_cell("ksi")],
        [wrap_cell("Steel tube yield (F<sub>y</sub>)"), wrap_cell(f"{mats.Fy_tube:.1f}"), wrap_cell("ksi")],
        [wrap_cell("Rebar yield (f<sub>y</sub>)"), wrap_cell(f"{mats.fy_bar:.1f}"), wrap_cell("ksi")],
        [wrap_cell("Steel modulus (E<sub>s</sub>)"), wrap_cell(f"{mats.Es:,.0f}"), wrap_cell("ksi")],
        [wrap_cell("Ultimate concrete strain (ε<sub>cu</sub>)"), wrap_cell(f"{mats.eps_cu:.4f}"), wrap_cell("—")],
        [wrap_cell("Confinement cap (round CFT)"), wrap_cell(f"{mats.conf_cap:.2f}·f'<sub>c</sub>"), wrap_cell("—")],
        [wrap_cell("Density factor (λ)"), wrap_cell(f"{mats.lambda_nw:.2f}"), wrap_cell("—")],
        [wrap_cell("Concrete unit weight"), wrap_cell(f"{mats.wc:.1f}"), wrap_cell("pcf")],
    ]

    mat_tbl = Table(mat_data, colWidths=[3.2*inch, 1.3*inch, 0.7*inch])
    mat_tbl.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.Color(0.7, 0.7, 0.7)),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 9),
        ('BOTTOMPADDING', (0, 0), (-1, 0), 8),
        ('BACKGROUND', (0, 1), (-1, -1), colors.Color(0.95, 0.95, 0.95)),
        ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ('ALIGN', (1, 1), (1, -1), 'RIGHT'),
        ('ALIGN', (2, 0), (2, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('LEFTPADDING', (0, 0), (-1, -1), 6),
        ('RIGHTPADDING', (0, 0), (-1, -1), 6),
        ('TOPPADDING', (0, 0), (-1, -1), 5),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
    ]))
    flow.append(mat_tbl)
    flow.append(Spacer(1, 10))

    # ========== 2. SECTION PROPERTIES ==========
    flow.append(Paragraph("<b>2. SECTION PROPERTIES</b>", styles['H2']))
    flow.append(Paragraph("<b>2.1 Areas</b>", styles['H3']))

    flow.append(Paragraph("Concrete core area:", styles['Normal']))
    flow.append(Paragraph(f"A<sub>c</sub> = π·D<sub>core</sub><super>2</super>/4 = π·({geom.D_core:.3f})<super>2</super>/4", styles['Equation']))
    flow.append(Paragraph(f"<b>A<sub>c</sub> = {Ac:.3f} in<super>2</super></b>", styles['Result']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph("Steel tube area:", styles['Normal']))
    flow.append(Paragraph(f"A<sub>s</sub> = π·(D<sub>o</sub><super>2</super> - D<sub>i</sub><super>2</super>)/4 = π·({geom.Do:.3f}<super>2</super> - {geom.Di:.3f}<super>2</super>)/4", styles['Equation']))
    flow.append(Paragraph(f"<b>A<sub>s</sub> = {As:.3f} in<super>2</super></b>", styles['Result']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph("Total rebar area:", styles['Normal']))
    flow.append(Paragraph(f"A<sub>b,tot</sub> = n·A<sub>bar</sub> = {geom.n_bars}·{geom.bar_area:.3f}", styles['Equation']))
    flow.append(Paragraph(f"<b>A<sub>b,tot</sub> = {Ab:.3f} in<super>2</super></b>", styles['Result']))
    flow.append(Spacer(1, 6))

    flow.append(Paragraph("<b>2.2 Moments of Inertia</b>", styles['H3']))

    flow.append(Paragraph("Concrete moment of inertia:", styles['Normal']))
    flow.append(Paragraph(f"I<sub>c</sub> = π·D<sub>core</sub><super>4</super>/64 = π·({geom.D_core:.3f})<super>4</super>/64", styles['Equation']))
    flow.append(Paragraph(f"<b>I<sub>c</sub> = {Ic:,.1f} in<super>4</super></b>", styles['Result']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph("Steel tube moment of inertia:", styles['Normal']))
    flow.append(Paragraph(f"I<sub>s</sub> = π·(D<sub>o</sub><super>4</super> - D<sub>i</sub><super>4</super>)/64 = π·({geom.Do:.3f}<super>4</super> - {geom.Di:.3f}<super>4</super>)/64", styles['Equation']))
    flow.append(Paragraph(f"<b>I<sub>s</sub> = {Is:,.1f} in<super>4</super></b>", styles['Result']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph("Rebar moment of inertia (parallel axis):", styles['Normal']))
    flow.append(Paragraph(f"I<sub>b</sub> = n·A<sub>bar</sub>·r<super>2</super> = {geom.n_bars}·{geom.bar_area:.3f}·({geom.bar_radius:.3f})<super>2</super>", styles['Equation']))
    flow.append(Paragraph(f"<b>I<sub>b</sub> = {Ib:,.1f} in<super>4</super></b>", styles['Result']))
    flow.append(Spacer(1, 3))

    I_tot = Ic + Is + Ib
    flow.append(Paragraph("Total transformed section:", styles['Normal']))
    flow.append(Paragraph(f"I<sub>tot</sub> = I<sub>c</sub> + I<sub>s</sub> + I<sub>b</sub> = {Ic:,.1f} + {Is:,.1f} + {Ib:,.1f}", styles['Equation']))
    flow.append(Paragraph(f"<b>I<sub>tot</sub> = {I_tot:,.1f} in<super>4</super></b>", styles['Result']))
    flow.append(Spacer(1, 10))

    # ========== 3. WHITNEY STRESS BLOCK ==========
    flow.append(Paragraph("<b>3. WHITNEY STRESS BLOCK PARAMETER (β<sub>1</sub>)</b>", styles['H2']))
    flow.append(Paragraph("Per ACI 318-19 Table 22.2.2.4.3:", styles['Normal']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph("β<sub>1</sub> = 0.85 for f'<sub>c</sub> ≤ 4 ksi", styles['Equation']))
    flow.append(Paragraph("β<sub>1</sub> = 0.85 - 0.05·(f'<sub>c</sub> - 4) for 4 &lt; f'<sub>c</sub> &lt; 8 ksi", styles['Equation']))
    flow.append(Paragraph("β<sub>1</sub> = 0.65 for f'<sub>c</sub> ≥ 8 ksi", styles['Equation']))
    flow.append(Spacer(1, 3))

    if mats.fc <= 4.0:
        flow.append(Paragraph(f"Since f'<sub>c</sub> = {mats.fc:.2f} ksi ≤ 4 ksi:", styles['Normal']))
        flow.append(Paragraph(f"<b>β<sub>1</sub> = 0.85</b>", styles['Result']))
    elif mats.fc >= 8.0:
        flow.append(Paragraph(f"Since f'<sub>c</sub> = {mats.fc:.2f} ksi ≥ 8 ksi:", styles['Normal']))
        flow.append(Paragraph(f"<b>β<sub>1</sub> = 0.65</b>", styles['Result']))
    else:
        flow.append(Paragraph(f"Since 4 &lt; f'<sub>c</sub> = {mats.fc:.2f} ksi &lt; 8:", styles['Normal']))
        flow.append(Paragraph(f"β<sub>1</sub> = 0.85 - 0.05·({mats.fc:.2f} - 4)", styles['Equation']))
        flow.append(Paragraph(f"<b>β<sub>1</sub> = {beta1:.3f}</b>", styles['Result']))
    flow.append(Spacer(1, 10))

    # ========== 4. STRENGTH REDUCTION FACTORS ==========
    flow.append(Paragraph("<b>4. STRENGTH REDUCTION FACTORS (φ)</b>", styles['H2']))
    flow.append(Paragraph("Per ACI 318-19 Table 21.2.2:", styles['Normal']))
    flow.append(Spacer(1, 4))

    phi_data = [
        [wrap_cell("<b>Load Type</b>"), wrap_cell("<b>φ Factor</b>"), wrap_cell("<b>Reference</b>")],
        [wrap_cell("Compression-controlled (tied)"), wrap_cell(f"{phi_c:.2f}"), wrap_cell("ACI 318-19 §21.2.2(a)")],
        [wrap_cell("Tension-controlled / flexure"), wrap_cell(f"{phi_b:.2f}"), wrap_cell("ACI 318-19 §21.2.2(c)")],
        [wrap_cell("Shear (steel)"), wrap_cell(f"{phi_v_steel:.2f}"), wrap_cell("AISC 360-22 §G1")],
        [wrap_cell("Shear (concrete)"), wrap_cell("0.75"), wrap_cell("ACI 318-19 §21.2.1(a)")],
    ]

    phi_tbl = Table(phi_data, colWidths=[2.5*inch, 1.2*inch, 1.8*inch])
    phi_tbl.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.Color(0.7, 0.7, 0.7)),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 9),
        ('BACKGROUND', (0, 1), (-1, -1), colors.Color(0.95, 0.95, 0.95)),
        ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ('ALIGN', (1, 0), (1, -1), 'CENTER'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('LEFTPADDING', (0, 0), (-1, -1), 6),
        ('RIGHTPADDING', (0, 0), (-1, -1), 6),
        ('TOPPADDING', (0, 0), (-1, -1), 5),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
    ]))
    flow.append(phi_tbl)
    flow.append(Spacer(1, 4))

    flow.append(Paragraph("For transition region (0.002 &lt; ε<sub>t</sub> &lt; 0.005):", styles['Normal']))
    flow.append(Paragraph("φ = φ<sub>comp</sub> + (ε<sub>t</sub> - 0.002)/(0.005 - 0.002)·(0.90 - φ<sub>comp</sub>)", styles['Equation']))
    flow.append(Spacer(1, 10))

    # ========== 5. AXIAL CAPACITY ==========
    flow.append(PageBreak())
    flow.append(Paragraph("<b>5. PURE AXIAL COMPRESSION CAPACITY</b>", styles['H2']))
    flow.append(Paragraph("Per AISC 360-22 §I2.1b for round CFT:", styles['Normal']))
    flow.append(Spacer(1, 4))

    flow.append(Paragraph("Nominal axial strength:", styles['Normal']))
    flow.append(Paragraph("P<sub>n</sub> = 0.95·f'<sub>c</sub>·A<sub>c</sub> + F<sub>y</sub>·A<sub>s</sub> + f<sub>y</sub>·A<sub>b</sub>", styles['Equation']))
    flow.append(Spacer(1, 3))

    Pn_calc = mats.conf_cap * mats.fc * Ac + mats.Fy_tube * As + mats.fy_bar * Ab
    flow.append(Paragraph(f"P<sub>n</sub> = 0.95·{mats.fc:.2f}·{Ac:.3f} + {mats.Fy_tube:.1f}·{As:.3f} + {mats.fy_bar:.1f}·{Ab:.3f}", styles['Equation']))
    flow.append(Paragraph(f"P<sub>n</sub> = {mats.conf_cap * mats.fc * Ac:.2f} + {mats.Fy_tube * As:.2f} + {mats.fy_bar * Ab:.2f}", styles['Equation']))
    flow.append(Paragraph(f"<b>P<sub>n</sub> = {Pn_calc:,.2f} kip</b>", styles['Result']))
    flow.append(Spacer(1, 6))

    flow.append(Paragraph(f"Design axial strength (φ = {phi_c:.2f}):", styles['Normal']))
    flow.append(Paragraph(f"φP<sub>n</sub> = {phi_c:.2f}·{Pn_calc:,.2f}", styles['Equation']))
    flow.append(Paragraph(f"<b>φP<sub>n</sub> = {axial_phiPn:,.2f} kip</b>", styles['HighlightResult']))
    flow.append(Spacer(1, 6))

    P80 = 0.80 * Pn_calc
    phiP80 = 0.80 * axial_phiPn
    flow.append(Paragraph(f"ACI 318-19 limit (0.80·P<sub>n,max</sub>):", styles['Normal']))
    flow.append(Paragraph(f"0.80·P<sub>n</sub> = {P80:,.2f} kip", styles['Equation']))
    flow.append(Paragraph(f"<b>0.80·φP<sub>n</sub> = {phiP80:,.2f} kip</b>", styles['HighlightResult']))
    flow.append(Spacer(1, 10))

    # ========== 6. SHEAR CAPACITY ==========
    flow.append(Paragraph("<b>6. SHEAR CAPACITY</b>", styles['H2']))

    flow.append(Paragraph("<b>6.1 Concrete Shear (ACI 318-19 §22.5.5.1)</b>", styles['H3']))
    flow.append(Paragraph("For circular sections without shear reinforcement:", styles['Normal']))
    flow.append(Spacer(1, 3))

    d_eff = shear_info['d']
    bw = shear_info['bw']
    Vc = shear_info['Vc_nom']
    phiVc = shear_info['phiVc']

    flow.append(Paragraph("Effective depth: d = 0.8·D (per ACI 318-19 §22.5.2.1)", styles['Normal']))
    flow.append(Paragraph(f"d = 0.8·{geom.D_core:.3f} = {d_eff:.3f} in", styles['Result']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph("Width: b<sub>w</sub> = D (diameter)", styles['Normal']))
    flow.append(Paragraph(f"b<sub>w</sub> = {bw:.3f} in", styles['Result']))
    flow.append(Spacer(1, 3))

    fc_psi = mats.fc * 1000
    flow.append(Paragraph("Concrete shear strength (one-way):", styles['Normal']))
    flow.append(Paragraph("V<sub>c</sub> = 2·λ·√(f'<sub>c</sub>[psi])·b<sub>w</sub>·d / 1000", styles['Equation']))
    flow.append(Paragraph(f"V<sub>c</sub> = 2·{mats.lambda_nw:.2f}·√{fc_psi:.0f}·{bw:.3f}·{d_eff:.3f} / 1000", styles['Equation']))
    flow.append(Paragraph(f"<b>V<sub>c</sub> = {Vc:.2f} kip</b>", styles['Result']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph("Design shear strength (φ = 0.75):", styles['Normal']))
    flow.append(Paragraph(f"<b>φV<sub>c</sub> = {phiVc:,.2f} kip</b>", styles['HighlightResult']))
    flow.append(Spacer(1, 8))

    flow.append(Paragraph("<b>6.2 Steel Tube Shear (AISC 360-22 §G2.1)</b>", styles['H3']))

    Av = shear_info['Av']
    Vn_steel = shear_info['Vn_steel_nom']
    phiV_steel = shear_info['phiV_steel']

    flow.append(Paragraph("Shear area for round HSS:", styles['Normal']))
    flow.append(Paragraph("A<sub>v</sub> ≈ 2·A<sub>s</sub>/π", styles['Equation']))
    flow.append(Paragraph(f"A<sub>v</sub> = 2·{As:.3f}/π = {Av:.3f} in<super>2</super>", styles['Result']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph("Nominal shear strength:", styles['Normal']))
    flow.append(Paragraph("V<sub>n</sub> = 0.6·F<sub>y</sub>·A<sub>v</sub>", styles['Equation']))
    flow.append(Paragraph(f"V<sub>n</sub> = 0.6·{mats.Fy_tube:.1f}·{Av:.3f} = {Vn_steel:.2f} kip", styles['Result']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph(f"Design shear strength (φ = {phi_v_steel:.2f}):", styles['Normal']))
    flow.append(Paragraph(f"<b>φV<sub>n</sub> = {phiV_steel:,.2f} kip</b>", styles['HighlightResult']))
    flow.append(Spacer(1, 8))

    flow.append(Paragraph("<b>6.3 Combined Shear Capacity</b>", styles['H3']))

    Vn_tot = shear_info['Vn_sum_nom']
    phiV_tot = shear_info['phiV_sum']

    flow.append(Paragraph("Total nominal shear:", styles['Normal']))
    flow.append(Paragraph(f"V<sub>n,total</sub> = V<sub>c</sub> + V<sub>n,steel</sub> = {Vc:.2f} + {Vn_steel:.2f}", styles['Equation']))
    flow.append(Paragraph(f"<b>V<sub>n,total</sub> = {Vn_tot:,.2f} kip</b>", styles['Result']))
    flow.append(Spacer(1, 3))

    flow.append(Paragraph("Total design shear:", styles['Normal']))
    flow.append(Paragraph(f"φV<sub>total</sub> = φV<sub>c</sub> + φV<sub>steel</sub> = {phiVc:.2f} + {phiV_steel:.2f}", styles['Equation']))
    flow.append(Paragraph(f"<b>φV<sub>total</sub> = {phiV_tot:,.2f} kip</b>", styles['HighlightResult']))
    flow.append(Spacer(1, 10))

    # ========== 7. KEY CONTROL POINTS ==========
    flow.append(PageBreak())
    flow.append(Paragraph("<b>7. P-M DIAGRAM KEY CONTROL POINTS</b>", styles['H2']))
    flow.append(Paragraph("Fiber-based strain compatibility analysis with ε<sub>cu</sub> = 0.003", styles['Normal']))
    flow.append(Spacer(1, 6))

    key_tbl_data = [
        [wrap_cell("<b>Pt</b>"), wrap_cell("<b>Description</b>"), wrap_cell("<b>c (in)</b>"),
         wrap_cell("<b>ε<sub>t</sub></b>"), wrap_cell("<b>P<sub>n</sub> (kip)</b>"), wrap_cell("<b>M<sub>n</sub> (kip-ft)</b>")]
    ]

    for r in key_rows:
        key_tbl_data.append([
            wrap_cell(str(r['Pt'])),
            wrap_cell(r['Definition']),
            wrap_cell("" if r['c'] is None else f"{r['c']:.3f}"),
            wrap_cell(f"{r['eps_t']:.5f}"),
            wrap_cell(f"{r['Pn']:,.1f}"),
            wrap_cell(f"{r['Mn']/12.0:,.1f}")
        ])

    key_tbl = Table(key_tbl_data, colWidths=[0.4*inch, 2.3*inch, 0.6*inch, 0.6*inch, 0.9*inch, 0.9*inch])
    key_tbl.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.Color(0.7, 0.7, 0.7)),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 8),
        ('BACKGROUND', (0, 1), (-1, -1), colors.Color(0.95, 0.95, 0.95)),
        ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ('ALIGN', (0, 0), (0, -1), 'CENTER'),
        ('ALIGN', (2, 1), (-1, -1), 'RIGHT'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('LEFTPADDING', (0, 0), (-1, -1), 4),
        ('RIGHTPADDING', (0, 0), (-1, -1), 4),
        ('TOPPADDING', (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
    ]))
    flow.append(key_tbl)
    flow.append(Spacer(1, 10))

    # ========== 8. PEAK MOMENTS ==========
    flow.append(Paragraph("<b>8. PEAK MOMENT CAPACITIES</b>", styles['H2']))

    peak_data = [
        [wrap_cell("<b>Description</b>"), wrap_cell("<b>Value (kip-ft)</b>"), wrap_cell("<b>Value (kip-in)</b>")],
        [wrap_cell("Nominal peak (M<sub>n,max</sub>)"), wrap_cell(f"{Mn_peak_nom/12.0:,.1f}"), wrap_cell(f"{Mn_peak_nom:,.1f}")],
        [wrap_cell("Factored peak (φM<sub>n,max</sub>) - nominal basis"), wrap_cell(f"{phiMn_peak_nom/12.0:,.1f}"), wrap_cell(f"{phiMn_peak_nom:,.1f}")],
        [wrap_cell("Factored peak - per-point φ basis"), wrap_cell(f"{Mn_peak_fac/12.0:,.1f}"), wrap_cell(f"{Mn_peak_fac:,.1f}")],
    ]

    peak_tbl = Table(peak_data, colWidths=[2.8*inch, 1.4*inch, 1.4*inch])
    peak_tbl.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.Color(0.7, 0.7, 0.7)),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.black),
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 9),
        ('BACKGROUND', (0, 1), (-1, -1), colors.Color(0.95, 0.95, 0.95)),
        ('GRID', (0, 0), (-1, -1), 1, colors.black),
        ('ALIGN', (1, 0), (-1, -1), 'RIGHT'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('LEFTPADDING', (0, 0), (-1, -1), 6),
        ('RIGHTPADDING', (0, 0), (-1, -1), 6),
        ('TOPPADDING', (0, 0), (-1, -1), 5),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
    ]))
    flow.append(peak_tbl)
    flow.append(Spacer(1, 10))

    # ========== 9. CODE REFERENCES ==========
    flow.append(Paragraph("<b>9. CODE REFERENCES</b>", styles['H2']))

    refs = [
        "<b>ACI 318-19:</b> Building Code Requirements for Structural Concrete",
        "   • §21.2: Strength reduction factors (φ factors)",
        "   • §22.2: Design assumptions for flexure",
        "   • §22.5: One-way shear provisions",
        "",
        "<b>AISC 360-22:</b> Specification for Structural Steel Buildings",
        "   • §I2.1b: Concrete-filled HSS compression members (round sections)",
        "   • §G2.1: Shear strength of round HSS members",
        "",
        "<b>ASTM A615:</b> Deformed and Plain Carbon-Steel Bars for Concrete Reinforcement",
        "",
        "<b>ASTM A500:</b> Cold-Formed Welded Carbon Steel Structural Tubing",
    ]

    for ref in refs:
        flow.append(Paragraph(ref, styles['Normal']))
    flow.append(Spacer(1, 10))

    # ========== 10. ASSUMPTIONS ==========
    flow.append(Paragraph("<b>10. ENGINEERING ASSUMPTIONS</b>", styles['H2']))

    assumptions = [
        "1. Concrete tension capacity = 0 (cracked section analysis)",
        f"2. Concrete compression capped at {mats.conf_cap}·f'<sub>c</sub> per AISC 360-22 §I2.1b for round CFT",
        f"3. Ultimate concrete strain ε<sub>cu</sub> = {mats.eps_cu:.4f} (ACI 318-19 §22.2.2.1)",
        "4. Steel (tube &amp; rebar) modeled as elastic-perfectly plastic",
        "5. Perfect bond between concrete, steel tube, and reinforcement",
        "6. Plane sections remain plane (strain compatibility)",
        "7. Shear analysis assumes no stirrups (concrete + tube only)",
        "8. Strength reduction factors per ACI 318-19 Table 21.2.2",
        "9. Fiber mesh convergence validated with circumferential divisions",
    ]

    for assumption in assumptions:
        flow.append(Paragraph(assumption, styles['Normal']))
    flow.append(Spacer(1, 12))

    # ========== FOOTER ==========
    flow.append(HRFlowable(width="100%", thickness=1.5, color=colors.black, spaceBefore=12))
    flow.append(Paragraph("<i>End of Report</i>", styles['Caption']))
    flow.append(Paragraph(f"<i>Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}</i>", styles['Caption']))

    # Build PDF with custom canvas
    doc.build(flow, canvasmaker=lambda *args, **kwargs: HeaderCanvas(*args, project=project, **kwargs))
    print(f"✓ PDF report → {out_pdf_path}")
# ========== INTERACTIVE INPUT FUNCTIONS ==========
def get_float_input(prompt: str, default: Optional[float] = None, min_val: Optional[float] = None, max_val: Optional[float] = None) -> float:
    """Get validated float input from user."""
    while True:
        default_str = f" [default: {default}]" if default is not None else ""
        user_input = input(f"{prompt}{default_str}: ").strip()

        if not user_input and default is not None:
            return default

        try:
            value = float(user_input)
            if min_val is not None and value < min_val:
                print(f"⚠️  Value must be >= {min_val}")
                continue
            if max_val is not None and value > max_val:
                print(f"⚠️  Value must be <= {max_val}")
                continue
            return value
        except ValueError:
            print("⚠️  Please enter a valid number")


def get_int_input(prompt: str, default: Optional[int] = None, min_val: Optional[int] = None, max_val: Optional[int] = None) -> int:
    """Get validated integer input from user."""
    while True:
        default_str = f" [default: {default}]" if default is not None else ""
        user_input = input(f"{prompt}{default_str}: ").strip()

        if not user_input and default is not None:
            return default

        try:
            value = int(user_input)
            if min_val is not None and value < min_val:
                print(f"⚠️  Value must be >= {min_val}")
                continue
            if max_val is not None and value > max_val:
                print(f"⚠️  Value must be <= {max_val}")
                continue
            return value
        except ValueError:
            print("⚠️  Please enter a valid integer")


def get_string_input(prompt: str, default: Optional[str] = None, allow_empty: bool = False) -> str:
    """Get string input from user."""
    default_str = f" [default: {default}]" if default is not None else ""
    user_input = input(f"{prompt}{default_str}: ").strip()

    if not user_input:
        if default is not None:
            return default
        elif allow_empty:
            return ""
        else:
            print("⚠️  This field cannot be empty")
            return get_string_input(prompt, default, allow_empty)

    return user_input


def get_yes_no(prompt: str, default: bool = True) -> bool:
    """Get yes/no input from user."""
    default_str = " [Y/n]" if default else " [y/N]"
    while True:
        user_input = input(f"{prompt}{default_str}: ").strip().lower()

        if not user_input:
            return default

        if user_input in ['y', 'yes']:
            return True
        elif user_input in ['n', 'no']:
            return False
        else:
            print("⚠️  Please enter 'y' or 'n'")


def get_bar_size() -> int:
    """Get rebar size with validation."""
    print("\nAvailable rebar sizes:")
    print("#3, #4, #5, #6, #7, #8, #9, #10, #11, #14, #18")

    while True:
        user_input = input("Enter bar size (e.g., 10 for #10) [default: 10]: ").strip()

        if not user_input:
            return 10

        try:
            bar_size = int(user_input)
            if bar_size in BAR_AREAS:
                return bar_size
            else:
                print(f"⚠️  Invalid bar size. Choose from: {list(BAR_AREAS.keys())}")
        except ValueError:
            print("⚠️  Please enter a valid integer")


def get_logo_path() -> Optional[str]:
    """Get logo path with validation."""
    if not get_yes_no("\nDo you want to include a company logo in the report?", default=False):
        return None

    while True:
        path = get_string_input("Enter full path to logo image (PNG, JPG, etc.)", allow_empty=True)

        if not path:
            return None

        if os.path.exists(path):
            # Check if it's an image file
            valid_extensions = ['.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff']
            if any(path.lower().endswith(ext) for ext in valid_extensions):
                return path
            else:
                print("⚠️  File must be an image (PNG, JPG, etc.)")
        else:
            print("⚠️  File not found. Please check the path.")
            if not get_yes_no("Try again?", default=True):
                return None


def collect_project_info() -> ProjectInfo:
    """Collect project information from user."""
    print("\n" + "="*60)
    print("PROJECT INFORMATION")
    print("="*60)

    project_name = get_string_input("Project name", default="Encased Drilled Pier Analysis")
    project_number = get_string_input("Project number", default="----")
    client = get_string_input("Client name", default="----")
    location = get_string_input("Project location", default="----")
    engineer = get_string_input("Engineer name", default="----")
    checker = get_string_input("Checker name", default="----")
    revision = get_string_input("Revision number", default="0")

    logo_path = get_logo_path()

    return ProjectInfo(
        project_name=project_name,
        project_number=project_number,
        client=client,
        location=location,
        engineer=engineer,
        checker=checker,
        date=time.strftime("%Y-%m-%d"),
        revision=revision,
        logo_path=logo_path
    )


def collect_materials() -> Materials:
    """Collect material properties from user."""
    print("\n" + "="*60)
    print("MATERIAL PROPERTIES")
    print("="*60)

    print("\n--- Concrete Properties ---")
    fc = get_float_input("Concrete compressive strength f'c (ksi)", default=4.0, min_val=2.5, max_val=15.0)

    # Calculate Ec using ACI 318-19 equation if user wants
    wc = get_float_input("Concrete unit weight (pcf)", default=150.0, min_val=90.0, max_val=160.0)
    Ec_calc = 33.0 * (wc ** 1.5) * math.sqrt(fc * 1000) / 1000.0  # ksi

    print(f"\nCalculated E_c (ACI 318-19): {Ec_calc:,.0f} ksi")
    if get_yes_no("Use calculated value?", default=True):
        Ec = Ec_calc
    else:
        Ec = get_float_input("Concrete modulus of elasticity E_c (ksi)", default=3600.0, min_val=1000.0)

    lambda_nw = get_float_input("Concrete density factor λ (1.0=normalweight, 0.85=sand-lightweight, 0.75=all-lightweight)",
                                default=1.0, min_val=0.75, max_val=1.0)

    print("\n--- Steel Tube Properties ---")
    print("Common values: A500 Gr.B = 46 ksi, A500 Gr.C = 50 ksi")
    Fy_tube = get_float_input("Steel tube yield strength F_y (ksi)", default=46.0, min_val=30.0, max_val=65.0)

    print("\n--- Reinforcing Bar Properties ---")
    print("Common values: Grade 60 = 60 ksi, Grade 75 = 75 ksi")
    fy_bar = get_float_input("Rebar yield strength f_y (ksi)", default=60.0, min_val=40.0, max_val=80.0)

    Es = get_float_input("Steel modulus of elasticity E_s (ksi)", default=29000.0, min_val=29000.0, max_val=29000.0)

    print("\n--- Strength Parameters ---")
    eps_cu = get_float_input("Ultimate concrete strain ε_cu (ACI 318-19: 0.003)", default=0.003, min_val=0.002, max_val=0.005)

    print("\nFor round CFT per AISC 360-22 §I2.1b, concrete compression is capped at 0.95·f'c")
    conf_cap = get_float_input("Confinement cap factor", default=0.95, min_val=0.85, max_val=1.0)

    return Materials(
        fc=fc,
        Ec=Ec,
        Es=Es,
        Fy_tube=Fy_tube,
        fy_bar=fy_bar,
        eps_cu=eps_cu,
        conf_cap=conf_cap,
        lambda_nw=lambda_nw,
        wc=wc
    )


def collect_geometry() -> Geometry:
    """Collect geometry properties from user."""
    print("\n" + "="*60)
    print("SECTION GEOMETRY")
    print("="*60)

    print("\n--- Concrete Core & Steel Tube ---")
    D_core = get_float_input("Concrete core diameter D_core (in)", default=36.0, min_val=12.0, max_val=120.0)

    print("\nCommon tube wall thicknesses: 0.250\", 0.375\", 0.500\", 0.625\", 0.750\"")
    t_tube = get_float_input("Steel tube wall thickness t (in)", default=0.500, min_val=0.125, max_val=2.0)

    print(f"\n✓ Outer diameter D_o = {D_core + 2*t_tube:.3f} in")

    print("\n--- Reinforcing Bars ---")
    n_bars = get_int_input("Number of longitudinal bars", default=12, min_val=4, max_val=48)
    bar_size = get_bar_size()

    print(f"\n✓ Bar area = {BAR_AREAS[bar_size]:.3f} in² each")
    print(f"✓ Total rebar area = {n_bars * BAR_AREAS[bar_size]:.3f} in²")

    cover = get_float_input("Clear cover to bar centerline (in)", default=3.0, min_val=1.5, max_val=6.0)

    bar_circle_radius = D_core/2 - cover
    if bar_circle_radius <= 0:
        print("⚠️  WARNING: Cover too large! Bars would be outside concrete core.")
        cover = get_float_input("Enter smaller cover value (in)", min_val=1.5, max_val=D_core/2 - 1.0)

    print(f"\n✓ Bars located at radius = {D_core/2 - cover:.3f} in from center")

    return Geometry(
        D_core=D_core,
        t_tube=t_tube,
        cover=cover,
        n_bars=n_bars,
        bar_size=bar_size
    )


def collect_mesh_control() -> MeshCtl:
    """Collect mesh control parameters (advanced users only)."""
    print("\n" + "="*60)
    print("MESH CONTROL (ADVANCED)")
    print("="*60)

    if not get_yes_no("\nUse advanced mesh settings? (default settings are recommended)", default=False):
        return MeshCtl()

    print("\nFiner meshes increase accuracy but take longer to compute.")
    n_theta = get_int_input("Circumferential divisions", default=180, min_val=36, max_val=360)
    n_rad_conc = get_int_input("Radial divisions (concrete)", default=36, min_val=12, max_val=60)
    n_ring_steel = get_int_input("Radial divisions (steel tube)", default=3, min_val=2, max_val=6)

    return MeshCtl(
        n_theta=n_theta,
        n_rad_conc=n_rad_conc,
        n_ring_steel=n_ring_steel
    )


def collect_analysis_options() -> Dict[str, Any]:
    """Collect analysis options."""
    print("\n" + "="*60)
    print("ANALYSIS OPTIONS")
    print("="*60)

    n_c = get_int_input("\nNumber of neutral axis positions to analyze", default=700, min_val=100, max_val=2000)

    spiral = get_yes_no("Use spiral reinforcement φ factors? (No = tied)", default=False)

    add_cloud = get_yes_no("Show point cloud on P-M diagram?", default=True)

    return {
        'n_c': n_c,
        'spiral': spiral,
        'add_cloud': add_cloud
    }


def show_summary(project: ProjectInfo, materials: Materials, geometry: Geometry, mesh: MeshCtl):
    """Display input summary for user confirmation."""
    print("\n" + "="*60)
    print("INPUT SUMMARY")
    print("="*60)

    print(f"\n--- Project ---")
    print(f"Name:     {project.project_name}")
    print(f"Number:   {project.project_number}")
    print(f"Client:   {project.client}")
    print(f"Location: {project.location}")
    print(f"Engineer: {project.engineer}")
    print(f"Logo:     {'Yes' if project.logo_path else 'No'}")

    print(f"\n--- Materials ---")
    print(f"f'c:      {materials.fc:.2f} ksi")
    print(f"E_c:      {materials.Ec:,.0f} ksi")
    print(f"F_y,tube: {materials.Fy_tube:.1f} ksi")
    print(f"f_y,bar:  {materials.fy_bar:.1f} ksi")

    print(f"\n--- Geometry ---")
    print(f"D_core:   {geometry.D_core:.3f} in")
    print(f"D_outer:  {geometry.Do:.3f} in")
    print(f"t_tube:   {geometry.t_tube:.3f} in")
    print(f"Bars:     {geometry.n_bars}-#{geometry.bar_size}")
    print(f"Cover:    {geometry.cover:.3f} in")

    # Calculate key properties
    Ac, As, Ab, Ic, Is, Ib = section_props(geometry)
    print(f"\n--- Computed Section Properties ---")
    print(f"A_concrete: {Ac:,.1f} in²")
    print(f"A_steel:    {As:,.1f} in²")
    print(f"A_rebar:    {Ab:,.2f} in²")
    print(f"I_total:    {Ic+Is+Ib:,.0f} in⁴")

    print(f"\n--- Mesh ---")
    print(f"Circumferential: {mesh.n_theta}")
    print(f"Radial (conc):   {mesh.n_rad_conc}")
    print(f"Radial (steel):  {mesh.n_ring_steel}")

    print("="*60)


def interactive_input() -> Tuple[ProjectInfo, Materials, Geometry, MeshCtl, Dict[str, Any]]:
    """Main interactive input function."""
    print("\n" + "█"*60)
    print("█" + " "*58 + "█")
    print("█" + "  ROUND CFT (ENCASED RC) ANALYSIS".center(58) + "█")
    print("█" + "  P-M Interaction Diagram Generator".center(58) + "█")
    print("█" + "  Per ACI 318-19 & AISC 360-22".center(58) + "█")
    print("█" + " "*58 + "█")
    print("█"*60)

    print("\nThis program will guide you through entering:")
    print("  • Project information")
    print("  • Material properties")
    print("  • Section geometry")
    print("  • Analysis options")
    print("\nPress Enter to use default values shown in [brackets]")

    input("\nPress Enter to continue...")

    # Collect all inputs
    project = collect_project_info()
    materials = collect_materials()
    geometry = collect_geometry()
    mesh = collect_mesh_control()
    options = collect_analysis_options()

    # Show summary and confirm
    show_summary(project, materials, geometry, mesh)

    if not get_yes_no("\nProceed with analysis?", default=True):
        print("\n❌ Analysis cancelled by user.")
        exit(0)

    return project, materials, geometry, mesh, options

# ========== MAIN EXECUTION ==========
# ========== MAIN EXECUTION ==========
def main():
    """Main analysis workflow with interactive input"""

    # ========== INTERACTIVE INPUT ==========
    project, materials, geometry, mesh, options = interactive_input()

    # ========== CREATE OUTPUT DIRECTORY ==========
    out_dir = make_output_dir()
    print(f"\n{'='*60}")
    print(f"Round CFT Analysis - Output Directory:")
    print(f"{out_dir}")
    print(f"{'='*60}\n")

    # ========== GENERATE P-M ENVELOPE ==========
    print("Generating P-M envelope...")
    Pn, Mn, Et = build_PM_cloud(geometry, materials, mesh, n_c=options['n_c'], return_eps=True)

    # ========== KEY POINTS ==========
    print("Calculating key control points...")
    key_rows = key_points_table(geometry, materials, mesh)

    # ========== PEAK MOMENTS ==========
    Pp_nom, Mp_nom = pick_peak_moment_point(Pn, Mn)
    Pf, Mf, phi_arr = factored_points_from_nominal(Pn, Mn, Et, spiral=options['spiral'])
    Pp_fac, Mp_fac = pick_peak_moment_point(Pf, Mf)

    # Factored version of nominal peak
    idx_nom_peak = int(np.argmax(np.abs(Mn)))
    phi_at_nom_peak = phi_arr[idx_nom_peak]
    Mp_nom_factored = phi_at_nom_peak * Mp_nom

    # ========== AXIAL CAPACITY ==========
    Ac, As, Ab, Ic, Is, Ib = section_props(geometry)
    Pn_axial = materials.conf_cap * materials.fc * Ac + materials.Fy_tube * As + materials.fy_bar * Ab
    phi_comp = PHI_COMP_SPIRAL if options['spiral'] else PHI_COMP_TIED
    phi_Pn_axial = phi_comp * Pn_axial

    # ========== SHEAR CAPACITY ==========
    shear_info = shear_strengths_aci_cft(geometry, materials)

    # ========== SAVE OUTPUTS ==========
    print("\nSaving outputs...")

    # CSV files
    save_key_points_csv(key_rows, (Pp_nom, Mp_nom), out_dir)
    save_PM_points_csv(Pn, Mn, out_dir, "PM_points.csv")

    # Section diagram
    fig_section = str(out_dir / "cft_section.png")
    plot_section(geometry, fig_section)
    print(f"✓ Section diagram → {fig_section}")

    # P-M envelope plot
    fig_envelope = str(out_dir / "pm_envelope.png")
    plot_PM_envelope(Pn, Mn, Et, fig_envelope, add_cloud=options['add_cloud'], spiral=options['spiral'])
    print(f"✓ P-M envelope → {fig_envelope}")

    # PDF report
    pdf_path = str(out_dir / "Encased_Drilled_Pier_Report.pdf")
    build_pdf(
        out_pdf_path=pdf_path,
        geom=geometry,
        mats=materials,
        project=project,
        phi_c=phi_comp,
        phi_b=PHI_FLEX_TENSION,
        phi_v_steel=0.90,
        fig_env_path=fig_envelope,
        fig_section_path=fig_section,
        key_rows=key_rows,
        axial_phiPn=phi_Pn_axial,
        shear_info=shear_info,
        Mn_peak_nom=Mp_nom,
        phiMn_peak_nom=Mp_nom_factored,
        Mn_peak_fac=Mp_fac
    )

    # ========== SUMMARY ==========
    print(f"\n{'='*60}")
    print("ANALYSIS SUMMARY")
    print(f"{'='*60}")
    print(f"Nominal P_n,max:      {Pn_axial:>12,.1f} kip")
    print(f"Design φP_n,max:      {phi_Pn_axial:>12,.1f} kip")
    print(f"0.80·φP_n,max:        {0.80*phi_Pn_axial:>12,.1f} kip")
    print(f"\nNominal M_n,peak:     {Mp_nom/12.0:>12,.1f} kip-ft")
    print(f"Factored peak:        {Mp_fac/12.0:>12,.1f} kip-ft")
    print(f"\nShear V_c (concrete): {shear_info['phiVc']:>12,.1f} kip")
    print(f"Shear V_n (steel):    {shear_info['phiV_steel']:>12,.1f} kip")
    print(f"Total φV:             {shear_info['phiV_sum']:>12,.1f} kip")
    print(f"{'='*60}\n")
    print(f"✅ Analysis complete! All files saved to:\n   {out_dir}\n")


if __name__ == "__main__":
    main()



████████████████████████████████████████████████████████████
█                                                          █
█              ROUND CFT (ENCASED RC) ANALYSIS             █
█             P-M Interaction Diagram Generator            █
█                Per ACI 318-19 & AISC 360-22              █
█                                                          █
████████████████████████████████████████████████████████████

This program will guide you through entering:
  • Project information
  • Material properties
  • Section geometry
  • Analysis options

Press Enter to use default values shown in [brackets]


KeyboardInterrupt: Interrupted by user